# పాఠం 01 - AI ఏజెంట్లకు పరిచయం

**AI ఏజెంట్లు ప్రారంభికుల కోసం** కోర్సులో మొదటి పాఠానికి స్వాగతం!

**AI ఏజెంట్** అనేది ఒక ప్రోగ్రామ్, ఇది పెద్ద భాషా మోడల్ (LLM) ను దాని తర్క యంత్రంగా ఉపయోగించి, వాస్తవ ప్రపంచంలో *చర్యలు* చేయగలదు — API లను పిలవడం, డేటాబేస్ లను విచారించడం, లేదా కోడ్ నడపడం ద్వారా — ఒక వినియోగదారుని తరఫున ఒక లక్ష్యాన్ని సాధించడానికి.

ఈ నోట్‌బుక్‌లో మీరు మీ మొదటి ఏజెంట్‌ను నిర్మించబోతున్నారు: ఒక **ప్రయాణ ఏజెంట్** ఇది సెలవు గమ్యస్థానాలను సూచిస్తుంది. ఈ పాఠం లో మీరు నేర్చుకునేది:

1. **Microsoft Agent Framework** ఉపయోగించి Azure AI Foundry ఏజెంట్ సర్వీసుతో ఎలా కనెక్ట్ కావాలో.
2. ఏజెంట్‌కు ఒక **సాధనం** ఇవ్వటం — ఏజెంట్ పిలవగల సాధారణ Python ఫంక్షన్.
3. ఏజెంట్‌ను నడిపించి దాని ప్రతిస్పందనను పరిశీలించడం.
4. ఏజెంట్ యొక్క ప్రతిస్పందనను టోకెన్-ద్వారా-టోకెన్ స్ట్రీమ్ చేయడం.


## సెటప్

ఈ నోటుబుక్‌ను চালించడానికి ముందు, మీరు ఖచ్చితంగా ఈ క్రింది వాటిని కలిగి ఉండాలి:

1. **డిప్లాయ్ చేసిన చాట్ మోడల్‌తో ఉన్న Azure AI Foundry ప్రాజెక్ట్** (ఉదా: `gpt-4o-mini`).
2. **Azure CLIలో లాగిన్ అయివి ఉండాలి** — మీ టెర్మినల్లో `az login` నడపండి.
3. **అవసరమైన ఎన్‌వైరిమెంట్ వేరియబుల్స్ సెట్ చేయాలి:**
   - `AZURE_AI_PROJECT_ENDPOINT` — మీ Azure AI Foundry ప్రాజెక్ట్ ఎండ్‌పాయింట్.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — మీ డిప్లాయ్ చేసిన మోడల్ పేరు.

కింది కౌశల్యం మీరు అవసరమైన Python ప్యాకేజీలను ఇన్‌స్టాల్ చేస్తుంది.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## మీ మొదటి ఏజెంట్‌ను సృష్టించడం

ఏజెంట్‌కు రెండు విషయాలు అవసరం:

- **సూచనలు** వాటి ద్వారా ఏజెంట్ *ఎవరో* మరియు *ఏ విధంగా ప్రవర్తించాలో* తెలిస్తాయి (సిస్టమ్ ప్రమ్ప్ట్).
- **ఉపకరణాలు** — Python ఫంక్షన్లు, వాటిని `@tool` అని అలంకరించబడినవి, అటువంటి ఏజెంట్ వాటిని కాల్ చేసి సమాచారాన్ని పొందగలదు లేదా పనులు చేసుకోగలదు.

కింద చమురు ప్రయాణాలకు ప్రముఖమైన మరియు ప్రాచుర్యం పొందిన గమ్యస్థలాల జాబితాను తిరిగిచ్చే ఒక సరళమైన ఉపకరణాన్ని నిర్వచించాము. ఒక వినియోగదారు ప్రయాణ సూచనలను కోరినప్పుడు, ఏజెంట్ ఈ ఉపకరణాన్ని ఉపయోగిస్తుంది.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## స్ట్రీమింగ్ స్పందనలు

ఇంకొంచెం అంతర్రాష్ట్ర అనుభవం కోసం మీరు ఏజెంట్ స్పందనను **స్ట్రీమ్** చేయవచ్చు. పూర్తి సమాధానాన్ని ఎదురుచూడకుండా, ఏజెంట్ ఉత్పత్తిచేస్తున్నప్పుడు టెక్స్ట్ భాగాలను పodelుస్తుంది. ఇది ప్రత్యేకంగా చాట్ ఇంటర్ఫేస్‌లలో ఉపయోగకరమై ఉంటుంది, అక్కడ మీరు రియల్ టైంలో అవుట్పుట్‌ను ప్రదర్శించాలని కోరుకుంటారు.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## సారాంశం

ఈ పాఠంలో మీరు నేర్చుకున్నది:

- `AzureAIProjectAgentProvider` ద్వారా Azure AI Foundry Agent Service కు కనెక్ట్ అయ్యే **ప్రొవైడర్ ను సృష్టించడం**.
- ఏజెంట్ మీ Python ఫังก్షన్‌లను పిలవగలుగుటకు `@tool` డెకరేటర్ ఉపయోగించి **ఒక టూల్ ను నిర్వచించడం**.
- యూజర్ సందేశంతో ఏజెంట్ ని **రన్ చేసి దాని స్పందనను ప్రింట్ చేయడం**.
- రియల్-టైమ్ అవుట్‌పుట్ కోసం **స్పందనలను స్ట్రీమ్ చేయడం**.

తదుపరి పాఠంలో ఏజెంటిక్ ఫ్రేమ్‌వర్క్‌లను ఎక్కువ వివరంగా పరిశీలించి ఏజెంట్లకు మరింత శక్తివంతమైన టూల్స్ మరియు బహుళ దశల తర్క సామర్థ్యాలను ఎలా ఇవ్వాలో నేర్చుకుంటాము.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
